In [14]:
from huggingface_hub import login

# IMPORTANT: HuggingFace Authentication Token Required
# This token allows access to gated models that require user agreement
# To get a token: Visit huggingface.co -> Settings -> Access Tokens -> Create new token
# Replace with your personal HuggingFace token for model access

from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32)
model.eval()


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [16]:
prompt = "The capital of England is"
# prompt = "The CEO of Tesla is"
# prompt = "The first president of USA is (surname only)"
# prompt = "The Vice-Chancellor (VC) of Brunel University London is (surname only)"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Get all intermediate activations (hidden states)
with torch.no_grad():
    outputs = model(input_ids, output_hidden_states=True)
    hidden_states = outputs.hidden_states  # tuple: [embeddings, layer1, layer2, ..., layerN]

# print("\nLogit Lens: Top-1 token at each layer (for last input token)")
# for i, act in enumerate(hidden_states[1:]):  # skip embeddings [0], start at layer1
#     act_last = act[0, -1, :]   # [batch 0, last token, :]
#     logits = model.lm_head(act_last)
#     top_id = logits.argmax().item()
#     token = tokenizer.decode([top_id])
#     print(f"Layer {i+1}: {token}")

token_pad = 12  # width for token text
print("\nLogit Lens: Top-5 tokens at each layer (for last input token):")
for i, act in enumerate(hidden_states[1:]):  # skip embedding layer [0]
    act_last = act[0, -1, :].float()   # Ensure float32 for stability
    logits = model.lm_head(act_last)
    probs = torch.softmax(logits, dim=-1)
    topk = torch.topk(probs, k=5)
    ids = topk.indices.tolist()
    ps = topk.values.tolist()
    tokens = [tokenizer.decode([id]).strip().replace('\n','') for id in ids]
    line = f"Layer {i+1:02d}:"
    for token, prob in zip(tokens, ps):
        line += f" {token.ljust(token_pad)}({prob:.5f})"
    print(line)


Logit Lens: Top-5 tokens at each layer (for last input token):
Layer 01: is          (0.00001) was         (0.00001) has         (0.00001) -is         (0.00001) Is          (0.00001)
Layer 02: is          (0.00001) was         (0.00001) is          (0.00001) تق          (0.00001) iso         (0.00001)
Layer 03: was         (0.00001) indeed      (0.00001) /is         (0.00001) acos        (0.00001) Es          (0.00001)
Layer 04: capital     (0.00001) osti        (0.00001) Capital     (0.00001) capital     (0.00001) otope       (0.00001)
Layer 05: currency    (0.00001) marty       (0.00001) utures      (0.00001) ़           (0.00001) stol        (0.00001)
Layer 06: capitals    (0.00001) @{          (0.00001) -names      (0.00001) capital     (0.00001) rael        (0.00001)
Layer 07: ptive       (0.00001) toi         (0.00001) opleft      (0.00001) tên         (0.00001) Debate      (0.00001)
Layer 08: stad        (0.00001) dém         (0.00001) français    (0.00001) toi         (0.00001